# Entrenamiento de sistemas de IA mediante Aprendizaje por Refuerzo para tareas de razonamiento  


## Modelo parametrizado

Un **modelo** es una función
$f_\theta:X \to Y$
que recibe una entrada $x\in X$ y produce una salida $f_\theta(x)\in Y$.  
El subíndice $\theta$ indica que el comportamiento de la función depende de un conjunto de parámetros.

- $X$: espacio de entradas (por ejemplo, vectores numéricos, imágenes o secuencias de tokens).
- $Y$: espacio de salidas (por ejemplo, una clase, un número, una secuencia de tokens o una acción).
- $\theta$: parámetros ajustables (pesos y sesgos).

Idea clave: “Aprender” significa elegir $\theta$ para que $f_\theta$ haga lo que queremos según algún criterio medible.


## Datos y objetivo de entrenamiento

En muchos casos se dispone de ejemplos:
$[D=\{(x_i,y_i)\}_{i=1}^n]$
donde $x_i$ es la entrada y $y_i$ es la salida deseada (la “respuesta correcta” o “etiqueta”). Esto es el escenario típico del aprendizaje supervisado.

Para medir qué tan bien se comporta el modelo se define una pérdida (o error), por ejemplo $\ell(\hat{y}, y)$, donde $\hat{y} = f_{\theta}(x)$ es la predicción del modelo. La pérdida total suele ser un promedio:

$$
L(\theta) = \frac{1}{n}\sum_{i=1}^{n} \ell\big(f_{\theta}(x_i), y_i\big).
$$

El entrenamiento busca:

$$
\theta^{*} = \arg\min_{\theta} L(\theta).
$$


## Optimización: descenso por gradiente

Queremos minimizar una función de pérdida $L(\theta)$ (donde $\theta$ son los parámetros).  
La idea del descenso por gradiente sale de una aproximación de primer orden (Taylor) alrededor de $\theta$.

**Aproximación local (Taylor de primer orden)**
Para un pequeño cambio $\Delta\theta$, se tiene:

$$
L(\theta + \Delta\theta)
\approx
L(\theta) + \nabla_{\theta}L(\theta)^\top \Delta\theta.
$$

Esto dice: *cerca de $\theta$, la pérdida cambia casi linealmente*, y la dirección que más aumenta $L$ es el gradiente $\nabla_\theta L(\theta)$.

**Elegir una dirección que disminuya $L$**

Si elegimos $\Delta\theta$ en la dirección opuesta al gradiente:

$$
\Delta\theta = -\eta \,\nabla_{\theta}L(\theta), \qquad \eta>0,
$$

entonces sustituyendo en Taylor:

$$
L(\theta + \Delta\theta)
\approx
L(\theta) + \nabla_{\theta}L(\theta)^\top\big(-\eta \nabla_{\theta}L(\theta)\big)
=
L(\theta) - \eta \,\|\nabla_{\theta}L(\theta)\|^2.
$$

Como $\|\nabla_{\theta}L(\theta)\|^2 \ge 0$ y $\eta>0$, esto sugiere que (para $\eta$ suficientemente pequeña) **la pérdida disminuye**.

De aquí sale la actualización estándar:

$$
\theta \leftarrow \theta - \eta\,\nabla_\theta L(\theta).
$$

donde:

- $\nabla_\theta L(\theta)$ es el vector de derivadas parciales de $L$ respecto a cada parámetro.
- $\eta>0$ es la tasa de aprendizaje (tamaño del paso).

En redes profundas, $L(\theta)$ es una composición de muchas funciones.  
Para ver el mecanismo matemático, consideremos una composición simple:

$$
f_{\theta}(x) = g_{\theta_2}\big(h_{\theta_1}(x)\big),
\qquad
\hat{y}=f_{\theta}(x),
\qquad
L(\theta)=\ell(\hat{y},y).
$$

Define la variable intermedia:

$$
z = h_{\theta_1}(x),
\qquad
\hat{y} = g_{\theta_2}(z).
$$

**Gradiente respecto a $\theta_2$ (bloque final)**

Por regla de la cadena:

$$
\nabla_{\theta_2} L
=
\frac{\partial L}{\partial \hat{y}}
\;\frac{\partial \hat{y}}{\partial \theta_2}
=
\Big(\nabla_{\hat{y}} \ell(\hat{y},y)\Big)\Big(\nabla_{\theta_2} g_{\theta_2}(z)\Big).
$$

**Gradiente respecto a $\theta_1$ (bloque inicial)** 

Aquí $L$ depende de $\theta_1$ a través de $z=h_{\theta_1}(x)$, entonces:

$$
\nabla_{\theta_1} L
=
\frac{\partial L}{\partial z}\;\frac{\partial z}{\partial \theta_1}.
$$

Pero

$$
\frac{\partial L}{\partial z}
=
\frac{\partial L}{\partial \hat{y}}\;\frac{\partial \hat{y}}{\partial z}
=
\Big(\nabla_{\hat{y}} \ell(\hat{y},y)\Big)\Big(\nabla_{z} g_{\theta_2}(z)\Big).
$$

Por tanto:

$$
\nabla_{\theta_1} L
=
\Big(\nabla_{\hat{y}} \ell(\hat{y},y)\Big)\Big(\nabla_{z} g_{\theta_2}(z)\Big)\Big(\nabla_{\theta_1} h_{\theta_1}(x)\Big).
$$

Interpretación: cada bloque aporta su derivada, y se van multiplicando “hacia atrás” (de la salida hacia la entrada).  
Eso es exactamente backpropagation.
